# Семинар 1. Packaging

За 50 минут пройдём путь от «папки с питоновскими файлами» до пакета,
который кто угодно поставит себе командой `pip install`. Потом 25 минут
делаете то же самое со своим пакетом и публикуете его.

| # | Раздел |
|---|---|
| 1 | Зачем вообще пакет |
| 2 | `pyproject.toml`: три секции |
| 3 | Собираем и смотрим, что получилось |
| 4 | Две вещи, которые ломаются молча |
| 5 | Публикация на TestPyPI |
| 6 | Задача |

Что глубже и подробнее — в соседнем [`extra.ipynb`](extra.ipynb). Там
версии, зависимости, устройство editable-установки и прочее. На семинар
не влезает, читать необязательно.

**Нужно прямо сейчас:** `uv` в PATH и аккаунт на
[test.pypi.org](https://test.pypi.org) с токеном.

Заготовка: временная папка под всё, что будем собирать. Запусти и забудь.

In [ ]:
import pathlib
import shutil
import subprocess
import sys
import tempfile
import zipfile

WORK = pathlib.Path(tempfile.mkdtemp(prefix="seminar01-"))
UV = shutil.which("uv")

print("песочница:", WORK)


def sh(command: str, cwd: pathlib.Path = WORK, quiet: bool = False) -> None:
    """Выполнить shell-команду в песочнице и показать её вывод."""
    result = subprocess.run(
        command, shell=True, cwd=cwd, capture_output=True, text=True
    )
    if quiet:
        return
    if result.stdout.strip():
        print(result.stdout, end="")
    if result.stderr.strip():
        print("--- stderr ---")
        print(result.stderr, end="")

## 1. Зачем вообще пакет

Обычный проект: `main.py` и папка `utils` рядом. Работает же.

In [ ]:
naive = WORK / "naive"
(naive / "utils").mkdir(parents=True)
(naive / "utils" / "text.py").write_text(
    "def shout(s):\n    return s.upper() + '!'\n", encoding="utf-8"
)
(naive / "main.py").write_text(
    "from utils.text import shout\n\nprint(shout('привет'))\n", encoding="utf-8"
)

sh(f"{sys.executable} main.py", cwd=naive)

Работает, потому что Python кладёт папку запускаемого скрипта в `sys.path`.
`utils` оказывается рядом и находится.

А теперь тот же импорт делает кто-то другой, из другой папки — ровно как
это произойдёт у однокурсника или на сервере:

In [ ]:
sh(f'{sys.executable} -c "import utils.text"', cwd=WORK)

`ModuleNotFoundError`. Код лежит на том же диске — и не находится.

Костыли, которые все пробовали:

```python
sys.path.append("../..")         # сломается при любом переносе
```

Правильный ответ один: **сделать код пакетом и установить в окружение**.
Тогда он находится откуда угодно, у него есть версия и способ доставки.

## 2. `pyproject.toml`: три секции

Всё описание пакета — в одном файле, и в нём три независимые части:

```toml
[build-system]   # ЧЕМ собирать
[project]        # ЧТО за пакет: имя, версия, описание, зависимости
[tool.*]         # настройки инструментов: backend'а, линтера, тестов
```

Разберём на живом примере. Сделаем пакет `demo_greet` — здоровается на
разных языках, фразы берёт из `.txt` рядом с кодом.

In [ ]:
DEMO = WORK / "demo-greet"
SRC = DEMO / "src" / "demo_greet"
(SRC / "data").mkdir(parents=True)

(SRC / "__init__.py").write_text('__version__ = "0.1.0"\n', encoding="utf-8")

# Фразы — обычный .txt внутри пакета. Запомни это место.
(SRC / "data" / "phrases.txt").write_text(
    "ru\tПривет\nen\tHello\nfr\tBonjour\n", encoding="utf-8"
)

(SRC / "core.py").write_text(
    """
from importlib.resources import files


def phrases() -> dict[str, str]:
    raw = files("demo_greet").joinpath("data", "phrases.txt").read_text(encoding="utf-8")
    return dict(line.split("\t") for line in raw.splitlines() if line)


def greet(name: str, lang: str = "ru") -> str:
    return f"{phrases()[lang]}, {name}!"
""".lstrip(),
    encoding="utf-8",
)

(SRC / "cli.py").write_text(
    """
import sys

from demo_greet.core import greet


def main() -> int:
    print(greet(sys.argv[1] if len(sys.argv) > 1 else "мир"))
    return 0
""".lstrip(),
    encoding="utf-8",
)

(DEMO / "README.md").write_text("# demo-greet\n\nДемо семинара 1.\n", encoding="utf-8")

sh(f"find {DEMO} -type f | sed 's|{DEMO}/||' | sort")

Код есть, пакета нет — нет `pyproject.toml`. Добавим минимальный:

In [ ]:
PYPROJECT = """
[build-system]
requires = ["hatchling>=1.27"]
build-backend = "hatchling.build"

[project]
name = "demo-greet-seminar"
version = "0.1.0"
description = "Демонстрационный пакет семинара 1"
readme = "README.md"
requires-python = ">=3.10"
dependencies = []

[project.scripts]
demo-greet = "demo_greet.cli:main"

[tool.hatch.build.targets.wheel]
packages = ["src/demo_greet"]
"""

(DEMO / "pyproject.toml").write_text(PYPROJECT.lstrip(), encoding="utf-8")
print("pyproject.toml записан — теперь это пакет")

Разберём по строчкам.

**`[build-system]`** — какой инструмент превратит исходники в пакет.
`uv`/`pip` про твой проект ничего не знают: они ставят указанный backend
во временное окружение и просят его собрать. Мы взяли `hatchling` —
современный дефолт для чистого Python. Бывают ещё `setuptools` (легаси и
C-расширения), `maturin` (Rust), и другие.

**`[project]`** — метаданные. Именно они попадут на страницу пакета.
`name` — под каким именем пакет лежит в индексе, `version`, `description`,
`readme`, `requires-python`, `dependencies`.

**`[project.scripts]`** — какие команды появятся в терминале. К этому
вернёмся в разделе 4.

**`[tool.hatch.build...]`** — настройка конкретного backend'а: где лежит
папка пакета. Это уже не про «что за пакет», а про «как его собирать».

> **Имя дистрибутива ≠ имя импорта.**
> Пакет называется `demo-greet-seminar`, а импортируется `demo_greet`.
> Это разные вещи, и совпадение — традиция, а не требование: ставишь
> `pillow` — импортируешь `PIL`, ставишь `scikit-learn` — импортируешь
> `sklearn`. В задаче семинара так же: дистрибутив
> `textstat-seminar-<твой-ник>`, импорт у всех один — `textstat_seminar`.

## 3. Собираем и смотрим, что получилось

In [ ]:
sh(f"{UV} build", cwd=DEMO)

Получилось два файла.

- **`.whl` (wheel)** — готовый к раскладке архив. `pip` его просто
  распаковывает в нужную папку. Быстро.
- **`.tar.gz` (sdist)** — исходники. `pip` будет **собирать** пакет у
  тебя на машине. Медленно, зато соберётся там, где готового колеса нет.

Публиковать надо оба. Подробности про имена и теги колёс — в `extra.ipynb`.

Wheel — это просто zip. Заглянем внутрь:

In [ ]:
wheel_path = next((DEMO / "dist").glob("*.whl"))
print(wheel_path.name, "\n")

with zipfile.ZipFile(wheel_path) as archive:
    for name in sorted(archive.namelist()):
        print(" ", name)

Две части: сам пакет и папка `*.dist-info` с метаданными. Посмотрим
метаданные — это ровно то, что покажет страница пакета на PyPI:

In [ ]:
with zipfile.ZipFile(wheel_path) as archive:
    metadata_name = next(n for n in archive.namelist() if n.endswith("/METADATA"))
    print(archive.read(metadata_name).decode())

Каждая строка приехала сюда из `[project]`. Пустой `Summary:` на странице
пакета — значит не заполнил `description`.

**Запомни эту команду**, она понадобится в задаче:

```bash
unzip -l dist/*.whl
```

## 4. Две вещи, которые ломаются молча

Дальше — самое важное на сегодня. Обе ошибки не видны при сборке: пакет
соберётся, опубликуется, и сломается **у пользователя**.

### Первая: точка входа

В конфиге написано:

```toml
[project.scripts]
demo-greet = "demo_greet.cli:main"
```

Что из этого получается? Поставим пакет и посмотрим.

In [ ]:
sh(f"{UV} venv {WORK}/venv", cwd=WORK, quiet=True)
sh(f"{UV} pip install --python {WORK}/venv/bin/python {DEMO}", cwd=WORK, quiet=True)

script = WORK / "venv" / "bin" / "demo-greet"
print("появился файл:", script, "\n")
print(script.read_text(encoding="utf-8"))

Никакой магии: обычный скрипт с shebang на Python из venv, который
импортирует твой модуль и зовёт функцию. Поэтому команда и работает
только в своём окружении — она физически лежит в его `bin/`.

In [ ]:
sh(f"{script} Даниэль")

Забыл `[project.scripts]` — сборка пройдёт, публикация пройдёт, а команды
у пользователя не будет. Это TODO 5 в задаче.

### Вторая: не-Python файлы

Наш пакет читает фразы из `data/phrases.txt`. После установки работает:

In [ ]:
sh(f'{WORK}/venv/bin/python -c "from demo_greet.core import phrases; print(phrases())"')

Работает, потому что backend у нас hatchling, и он кладёт в колесо всё
содержимое папки пакета.

Теперь возьмём **тот же самый код и тот же `[project]`**, поменяем только
backend на setuptools:

In [ ]:
SETUP_VARIANT = WORK / "demo-greet-setuptools"
shutil.copytree(DEMO, SETUP_VARIANT, ignore=shutil.ignore_patterns("dist", "build"))

config = (SETUP_VARIANT / "pyproject.toml").read_text(encoding="utf-8")
config = config.replace('requires = ["hatchling>=1.27"]', 'requires = ["setuptools>=68"]')
config = config.replace(
    'build-backend = "hatchling.build"', 'build-backend = "setuptools.build_meta"'
)
config = config.split("[tool.hatch.build.targets.wheel]")[0]
config += '[tool.setuptools.packages.find]\nwhere = ["src"]\n'
(SETUP_VARIANT / "pyproject.toml").write_text(config, encoding="utf-8")

sh(f"{UV} build --wheel", cwd=SETUP_VARIANT, quiet=True)

setup_wheel = next((SETUP_VARIANT / "dist").glob("*.whl"))
with zipfile.ZipFile(setup_wheel) as archive:
    for name in sorted(n for n in archive.namelist() if n.startswith("demo_greet/")):
        print(" ", name)

**`phrases.txt` в колесе нет.**

Локально всё работало — исходники лежали рядом. У пользователя —
`FileNotFoundError`, причём не при установке, а при первом вызове функции,
которая читает данные.

Лечится настройкой backend'а:

```toml
# setuptools
[tool.setuptools.package-data]
demo_greet = ["data/*.txt"]

# hatchling — само, если указана папка пакета
[tool.hatch.build.targets.wheel]
packages = ["src/demo_greet"]
```

Это TODO 6 в задаче.

### Главный вывод семинара

Включение не-Python файлов — не часть Python и не часть стандарта на
`pyproject.toml`. Это политика конкретного backend'а, и она у всех разная.

Отсюда правило:

> **«Собралось» ≠ «собралось правильно».**
> Единственная надёжная проверка — посмотреть, что внутри:
> `unzip -l dist/*.whl`

## 5. Публикация на TestPyPI

**TestPyPI** ([test.pypi.org](https://test.pypi.org)) — копия PyPI для
тренировки. Отдельная база пользователей, отдельные токены, периодически
чистится. Ломать там ничего не страшно.

### Токен

1. Регистрация: <https://test.pypi.org/account/register/>
2. Подтвердить email, включить 2FA — без этого загрузка не работает.
3. Account settings → API tokens → Add API token, scope «Entire account».
4. Скопировать **вместе** с префиксом `pypi-`. Показывают один раз.

### Загрузка

```bash
uv build
unzip -l dist/*.whl        # сначала посмотреть!

export UV_PUBLISH_URL=https://test.pypi.org/legacy/
export UV_PUBLISH_TOKEN=pypi-AgENdGVzdC5weXBpLm9yZw...
uv publish
```

Токен — через переменную окружения, а не аргументом команды: иначе он
останется в истории shell, которую легко показать на проекторе.

### Проверка с нуля

```bash
cd /tmp && uv venv proba && cd proba
uv pip install \
  --index-url https://test.pypi.org/simple/ \
  --extra-index-url https://pypi.org/simple/ \
  demo-greet-seminar
```

`--extra-index-url` нужен потому, что на TestPyPI нет копий обычных
пакетов — зависимости оттуда не поставятся.

### Три грабли, на которые наступят все

**Версию нельзя перезалить. Никогда.** `400 File already exists`. Даже
если удалить релиз через сайт, имя файла останется занятым. Выход один —
поднять версию. Это защита от подмены: однажды скачанный
`pkg-1.2.3-py3-none-any.whl` у всех одинаковый.

Отсюда и порядок: сначала `unzip -l`, потом `publish`.

**403 при загрузке.** Не подтверждён email, не включена 2FA, или токен
скопирован без префикса `pypi-`.

**Залил на pypi.org вместо test.pypi.org.** Забыл `UV_PUBLISH_URL`.
Удалить нельзя, можно только yank — версия останется, но перестанет
ставиться по умолчанию.

## 6. Задача

Условие — в [`task/README.md`](task/README.md).

В `task/textstat_template/` лежит готовый код пакета `textstat_seminar`
и `pyproject.toml` с шестью TODO. Скучные поля уже заполнены. Надо:

1. закрыть TODO,
2. собрать и **посмотреть в колесо**,
3. опубликовать на TestPyPI под именем `textstat-seminar-<твой-ник>`,
4. проверить, что ставится с нуля.

Проверка:

```bash
make check-pub USERNAME=<твой-ник>
```

TODO 5 и 6 — ровно те две вещи из раздела 4, которые ломаются молча.

Если останется время или интерес — [`extra.ipynb`](extra.ipynb): версии,
зависимости, src-layout, устройство editable-установки, публикация из CI
без токенов.

In [ ]:
shutil.rmtree(WORK, ignore_errors=True)
print("песочница удалена:", WORK)